# Task 3: 30-Day Spending Prediction (Regression — NMAE Score)

**Objective**: Predict each player's 30-day spending (`spending_30d`), evaluated by **Normalized MAE Score**:
$$\text{NMAE Score} = \frac{1}{1 + \text{MAE} / \text{mean}(|y|)}$$

### Pipeline Summary
1. **Essential EDA** — Show the zero-inflated, heavy-tailed target (~48% zeros; whales drive ~80% of total spending) and verify exact algebraic relationships among spending/transaction columns.
2. **Preprocessing** — Algebraically recover missing values, engineer spending-rate/log features, and impute missing `segment` with a train-only XGBoost classifier.
3. **10-Seed Two-Stage Hurdle Model**:
   - **Stage 1**: 10-seed × 5-fold LightGBM binary classifier predicting payer probability $p = P(y > 0 | x)$.
   - **Stage 2**: 10-seed × 5-fold CatBoost depth-4 `MultiQuantile` regressor on positive spenders predicting $\log(y) - \text{offset}(x)$ across 11 quantiles $\alpha \in [0.02, 0.50]$.
4. **MAE-Optimal Point Prediction** — Combine $p$ and the positive quantiles via the exact overall median: $0$ if $p \le 0.5$, else the conditional quantile $q = 1 - \frac{1}{2p}$.

In [1]:
import os
import glob
import time
import warnings
import numpy as np
import pandas as pd

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")

def find_data_dir(subdir: str) -> str:
    for p in [os.path.join("..", "cpe342-kaggle-main", subdir), os.path.join("cpe342-kaggle-main", subdir)]:
        if os.path.isdir(p):
            return os.path.abspath(p)
    hits = glob.glob(f"/kaggle/input/**/{subdir}/train.csv", recursive=True)
    if hits:
        return os.path.dirname(hits[0])
    raise FileNotFoundError(f"Dataset not found for {subdir}. Expected in cpe342-kaggle-main/{subdir}.")

DATA_DIR = find_data_dir("03_regression")
OUT_CSV = "task3_predictions.csv"
TARGET = "spending_30d"
N_JOBS = os.cpu_count() or 4
ALPHAS = np.array([0.02, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50])
SEEDS = list(range(10))

tr = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
te = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
y = tr[TARGET].values.astype(float)
test_ids = np.array([f"ANS{i+1:05d}" for i in range(len(te))])

print(f"Train: {tr.shape} | Test: {te.shape}")

Train: (104000, 34) | Test: (25889, 33)


## 1. Essential EDA & Design Decisions

1. **Zero-Inflation & Heavy Tail**: Nearly half of all accounts spend `0`, while the top segment (`segment == 3`, whales) accounts for the vast majority of total spending $\rightarrow$ **Decision: use a Two-Stage Hurdle Model and model the positive part on the log-residual scale (`log(y) - prev_offset`).**
2. **Exact Algebraic Identities**: Check `historical_spending == total_transactions * avg_transaction_value` and `purchases_on_discount == total_transactions * discount_rate_used` on complete rows $\rightarrow$ **Decision: algebraically fill missing values before modeling.**
3. **Why Multi-Quantile Hurdle?**: Under L1/MAE loss, the Bayes-optimal point prediction is the **overall median**, which equals $0$ when $p \le 0.5$ and the conditional quantile $q = 1 - \frac{1}{2p} \in (0, 0.5]$ of the positive distribution when $p > 0.5$.

In [2]:
# 1. Target zero-inflation and segment spending share
print(f"Zero-spending rate: {(y == 0).mean()*100:.2f}% | Positive median: {np.median(y[y > 0]):,.1f} | Max: {y.max():,.1f}")
seg_share = tr.groupby("segment")[TARGET].agg(["count", "mean", "sum"])
seg_share["spend_share_%"] = 100 * seg_share["sum"] / tr[TARGET].sum()
print("\nSpending by observed segment:")
print(seg_share.round(2))

# 2. Verify exact algebraic identities on observed rows
m1 = tr[["historical_spending", "total_transactions", "avg_transaction_value"]].notna().all(1)
err1 = (tr.loc[m1, "historical_spending"] - tr.loc[m1, "total_transactions"] * tr.loc[m1, "avg_transaction_value"]).abs()
m2 = tr[["purchases_on_discount", "total_transactions", "discount_rate_used"]].notna().all(1)
err2 = (tr.loc[m2, "purchases_on_discount"] - tr.loc[m2, "total_transactions"] * tr.loc[m2, "discount_rate_used"]).abs()
print(f"\nIdentity check | historical_spending = tx * atv: median err={err1.median():.6f}, exact(<1e-2)={(err1 < 1e-2).mean()*100:.2f}%")
print(f"Identity check | purchases_on_discount = tx * disc_rate: median err={err2.median():.6f}, exact(<1e-2)={(err2 < 1e-2).mean()*100:.2f}%")

Zero-spending rate: 48.12% | Positive median: 2,314.1 | Max: 300,000.0

Spending by observed segment:
         count      mean           sum  spend_share_%
segment                                              
0.0      39241     73.96  2.902183e+06           0.27
1.0      24483    944.37  2.312102e+07           2.12
2.0      19762   2283.85  4.513335e+07           4.13
3.0      14579  65357.70  9.528499e+08          87.28

Identity check | historical_spending = tx * atv: median err=0.000000, exact(<1e-2)=99.00%
Identity check | purchases_on_discount = tx * disc_rate: median err=0.000000, exact(<1e-2)=100.00%


## 2. Preprocessing: Algebraic Recovery, Feature Engineering & Train-Only Segment Imputer

In [3]:
def nmae(yt: np.ndarray, yp: np.ndarray) -> float:
    return float(1.0 / (1.0 + np.abs(yt - yp).sum() / yt.sum()))


def exact_fill(d: pd.DataFrame) -> pd.DataFrame:
    d = d.copy()
    for _ in range(3):
        h, tx, av = d["historical_spending"], d["total_transactions"], d["avg_transaction_value"]
        po, dr = d["purchases_on_discount"], d["discount_rate_used"]
        rules = [
            ("historical_spending", h.isna() & tx.notna() & av.notna(), tx * av),
            ("total_transactions", tx.isna() & h.notna() & av.notna() & (av > 0), h / av),
            ("avg_transaction_value", av.isna() & h.notna() & tx.notna() & (tx > 0), h / tx),
            ("purchases_on_discount", po.isna() & tx.notna() & dr.notna(), tx * dr),
            ("total_transactions", tx.isna() & po.notna() & dr.notna() & (dr > 0), po / dr),
            ("discount_rate_used", dr.isna() & po.notna() & tx.notna() & (tx > 0), po / tx),
        ]
        for col, m, val in rules:
            m = m & d[col].isna()
            d.loc[m, col] = val[m]
    return d


def add_feats(d: pd.DataFrame) -> pd.DataFrame:
    d = d.copy()
    e = 1.0
    d["hist_per_day"] = d["historical_spending"] / (d["account_age_days"] + e)
    d["tx_per_day"] = d["total_transactions"] / (d["account_age_days"] + e)
    d["exp_monthly_spend"] = d["avg_transaction_value"] * d["purchase_frequency"]
    d["prev_vs_hist"] = d["prev_month_spending"] / (d["historical_spending"] + e)
    d["prev_x_freq"] = d["prev_month_spending"] * d["purchase_frequency"]
    d["recency_ratio"] = d["days_since_last_purchase"] / (d["days_since_last_login"] + e)
    d["play_per_session"] = d["total_playtime_hours"] / (d["sessions_per_week"] + e)
    for c in ["historical_spending", "prev_month_spending", "avg_transaction_value"]:
        d[f"log_{c}"] = np.log1p(d[c].clip(lower=0))
    d["n_missing"] = d.isna().sum(axis=1)
    return d


def add_log_ratio_feats(X: pd.DataFrame, Xt: pd.DataFrame):
    X, Xt = X.copy(), Xt.copy()
    for Z in (X, Xt):
        lp, lh = Z["log_prev_month_spending"], Z["log_historical_spending"]
        Z["lprev_minus_lhist"] = lp - lh
        Z["l_atv_x_freq"] = np.log1p(Z["avg_transaction_value"].clip(lower=0) * Z["purchase_frequency"])
        Z["l_hist_per_tx"] = np.log1p(Z["historical_spending"] / (Z["total_transactions"] + 1))
        Z["l_hist_per_age"] = np.log1p(Z["hist_per_day"])
    return X, Xt


def prev_offset(Z: pd.DataFrame) -> np.ndarray:
    lp = Z["log_prev_month_spending"].values
    o = np.where(np.isnan(lp), Z["log_historical_spending"].values - 1.05, lp)
    return np.where(np.isnan(o), 6.5, o)


tr_f, te_f = exact_fill(tr), exact_fill(te)
X = add_feats(tr_f.drop(columns=[TARGET]))
Xt = add_feats(te_f)
FE = [c for c in Xt.columns if c != "player_id"]
X, Xt = X[FE].copy(), Xt[FE].copy()

# Train-only XGBoost imputer for missing segment + whale probability feature
SF = [c for c in FE if c != "segment"]
known = X["segment"].notna().values
sc = xgb.XGBClassifier(
    objective="multi:softprob", n_estimators=4000, learning_rate=0.02, early_stopping_rounds=100,
    max_depth=6, subsample=0.8, colsample_bytree=0.7, tree_method="hist", random_state=42, n_jobs=N_JOBS
)
i_a, i_b = train_test_split(np.where(known)[0], test_size=0.1, random_state=42, stratify=X["segment"].values[known])
sc.fit(
    X.iloc[i_a][SF], X["segment"].values[i_a].astype(int),
    eval_set=[(X.iloc[i_b][SF], X["segment"].values[i_b].astype(int))], verbose=False
)
print(f"Segment imputer holdout accuracy: {(sc.predict(X.iloc[i_b][SF]) == X['segment'].values[i_b]).mean():.4f}")

for D in (X, Xt):
    miss = D["segment"].isna().values
    P = np.zeros((len(D), 4))
    P[~miss, D.loc[~miss, "segment"].astype(int).values] = 1.0
    P[miss] = sc.predict_proba(D.loc[miss, SF])
    D["seg_imputed"] = miss.astype(int)
    D.loc[miss, "segment"] = P[miss].argmax(1)
    D["p_whale"] = P[:, 3]

X_cb, Xt_cb = add_log_ratio_feats(X, Xt)

# 5-fold StratifiedKFold on zero vs 4 payer quartiles
pos = y > 0
strat = np.where(pos, 1 + pd.qcut(pd.Series(y).where(pos).rank(method="first"), 4, labels=False).fillna(-1).values, 0).astype(int)
FOLDS = list(StratifiedKFold(5, shuffle=True, random_state=42).split(np.zeros(len(y)), strat))
print(f"Prepared features: Stage-1={X.shape}, Stage-2={X_cb.shape}")

Segment imputer holdout accuracy: 0.9292


Prepared features: Stage-1=(104000, 45), Stage-2=(104000, 49)


## 3. Stage 1: 10-Seed LightGBM Payer Classifier $P(y > 0 | x)$

In [4]:
n, nt = len(y), len(Xt)
P_oofs, P_tests = [], []
t0 = time.time()

for seed in SEEDS:
    Op = np.zeros(n)
    Tp = np.zeros(nt)
    for k, (tr_i, va_i) in enumerate(FOLDS):
        Xa, ya = X.iloc[tr_i], (y[tr_i] > 0).astype(int)
        i, j = train_test_split(np.arange(len(ya)), test_size=0.1, random_state=seed, stratify=ya)
        m = lgb.LGBMClassifier(
            objective="binary", learning_rate=0.03, num_leaves=63, min_child_samples=40,
            subsample=0.8, subsample_freq=1, colsample_bytree=0.7, reg_lambda=2.0,
            n_estimators=10000, verbose=-1, n_jobs=N_JOBS, random_state=seed
        )
        m.fit(
            Xa.iloc[i], ya[i],
            eval_set=[(Xa.iloc[j], ya[j])],
            callbacks=[lgb.early_stopping(100, verbose=False)],
        )
        Op[va_i] = m.predict_proba(X.iloc[va_i])[:, 1]
        Tp += m.predict_proba(Xt)[:, 1] / 5
    P_oofs.append(Op)
    P_tests.append(Tp)
    print(f"  [LGBM P(y>0)] seed {seed}: OOF AUC={roc_auc_score(y > 0, Op):.5f} ({time.time()-t0:.0f}s)")

P_oof = np.mean(P_oofs, axis=0)
P_test = np.mean(P_tests, axis=0)
print(f"[Stage 1: 10-seed P(y>0)] OOF AUC: {roc_auc_score(y > 0, P_oof):.5f}")

  [LGBM P(y>0)] seed 0: OOF AUC=0.78397 (45s)


  [LGBM P(y>0)] seed 1: OOF AUC=0.78408 (71s)


  [LGBM P(y>0)] seed 2: OOF AUC=0.78370 (106s)


  [LGBM P(y>0)] seed 3: OOF AUC=0.78435 (139s)


  [LGBM P(y>0)] seed 4: OOF AUC=0.78420 (168s)


  [LGBM P(y>0)] seed 5: OOF AUC=0.78418 (185s)


  [LGBM P(y>0)] seed 6: OOF AUC=0.78405 (207s)


  [LGBM P(y>0)] seed 7: OOF AUC=0.78383 (253s)


  [LGBM P(y>0)] seed 8: OOF AUC=0.78440 (303s)


  [LGBM P(y>0)] seed 9: OOF AUC=0.78398 (325s)
[Stage 1: 10-seed P(y>0)] OOF AUC: 0.78439


## 4. Stage 2: 10-Seed CatBoost MultiQuantile Regressor (Positive Spenders)

Predicts 11 conditional quantiles of $\log(y) - \text{prev\_offset}(x)$ on positive spenders (`depth=4`, `lr=0.05`, `l2_leaf_reg=3`), then averages across seeds using the geometric mean on the original spending scale.

In [5]:
loss_str = "MultiQuantile:alpha=" + ",".join(str(x) for x in ALPHAS)
Q_oofs, Q_tests = [], []
t0 = time.time()

for seed in SEEDS:
    OQ = np.zeros((n, len(ALPHAS)))
    TQ = np.zeros((nt, len(ALPHAS)))
    for k, (tr_i, va_i) in enumerate(FOLDS):
        Xa, ya = X_cb.iloc[tr_i], y[tr_i]
        p_mask = ya > 0
        Xp = Xa[p_mask]
        r = np.log(ya[p_mask]) - prev_offset(Xp)
        i, j = train_test_split(np.arange(len(r)), test_size=0.1, random_state=seed)
        m = CatBoostRegressor(
            loss_function=loss_str, iterations=20000, learning_rate=0.05, depth=4,
            l2_leaf_reg=3.0, border_count=254, random_strength=1.0, bagging_temperature=1.0,
            rsm=1.0, grow_policy="SymmetricTree", od_type="Iter", od_wait=200,
            random_seed=seed, thread_count=N_JOBS, verbose=False
        )
        m.fit(Xp.iloc[i], r[i], eval_set=(Xp.iloc[j], r[j]), use_best_model=True)
        OQ[va_i] = np.exp(np.sort(m.predict(X_cb.iloc[va_i]), axis=1) + prev_offset(X_cb.iloc[va_i])[:, None])
        TQ += np.exp(np.sort(m.predict(Xt_cb), axis=1) + prev_offset(Xt_cb)[:, None]) / 5
    Q_oofs.append(OQ)
    Q_tests.append(TQ)
    print(f"  [CatBoost MultiQuantile] seed {seed} done ({(time.time()-t0)/60:.1f} min)")

Q_oof = np.exp(np.mean([np.log(q) for q in Q_oofs], axis=0))
Q_test = np.exp(np.mean([np.log(q) for q in Q_tests], axis=0))

  [CatBoost MultiQuantile] seed 0 done (10.0 min)


  [CatBoost MultiQuantile] seed 1 done (21.0 min)


  [CatBoost MultiQuantile] seed 2 done (29.4 min)


  [CatBoost MultiQuantile] seed 3 done (37.9 min)


  [CatBoost MultiQuantile] seed 4 done (46.7 min)


  [CatBoost MultiQuantile] seed 5 done (56.2 min)


  [CatBoost MultiQuantile] seed 6 done (64.2 min)


  [CatBoost MultiQuantile] seed 7 done (73.5 min)


  [CatBoost MultiQuantile] seed 8 done (80.1 min)


  [CatBoost MultiQuantile] seed 9 done (87.1 min)


## 5. MAE-Optimal Hurdle Median & Save Predictions

In [6]:
def hurdle_median(p: np.ndarray, Q: np.ndarray, alphas: np.ndarray, cap: float = 300000.0) -> np.ndarray:
    pp = np.clip(p, 1e-6, 1.0)
    q = np.clip(1.0 - 1.0 / (2.0 * pp), alphas[0], alphas[-1])
    Q_sorted = np.sort(Q, axis=1)
    j = np.clip(np.searchsorted(alphas, q) - 1, 0, len(alphas) - 2)
    a0, a1 = alphas[j], alphas[j + 1]
    r = np.arange(len(q))
    v = Q_sorted[r, j] + (q - a0) / (a1 - a0) * (Q_sorted[r, j + 1] - Q_sorted[r, j])
    return np.where(pp > 0.5, np.clip(v, 0.0, cap), 0.0)


oof_pred = hurdle_median(P_oof, Q_oof, ALPHAS)
test_pred = hurdle_median(P_test, Q_test, ALPHAS)

fold_scores = [nmae(y[b], oof_pred[b]) for _, b in FOLDS]
print(f"Final 10-Seed Hurdle OOF NMAE: {nmae(y, oof_pred):.5f} (fold mean {np.mean(fold_scores):.5f} +- {np.std(fold_scores):.5f})")

sub_t3 = pd.DataFrame({"id": test_ids, "pred": test_pred})
sub_t3.to_csv(OUT_CSV, index=False, lineterminator="\n")
print(f"Saved {OUT_CSV} | rows={len(sub_t3)} | test zero rate={(test_pred == 0).mean():.4f} | mean={test_pred.mean():.2f}")
sub_t3.head()

Final 10-Seed Hurdle OOF NMAE: 0.79535 (fold mean 0.79540 +- 0.00761)
Saved task3_predictions.csv | rows=25889 | test zero rate=0.5042 | mean=10429.87


,id,pred
0,ANS00001,0.000000
1,ANS00002,2188.581164
2,ANS00003,0.000000
3,ANS00004,0.000000
4,ANS00005,0.000000
